# A100 validation of a Gaussian photonic sensing model

This is a **classical simulation**, not quantum computing, optical hardware, SRI error annihilation, or a demonstration of quantum advantage. See `docs/cv-photonics-protocol.md` in the repository for the physical model and limitations. NumPy and Julia run on CPU; PyTorch, JAX and Triton must run on an A100. Same simulated shots are used for backend agreement, not independent evidence.

Select **Runtime → Change runtime type → A100 GPU**. Download the four current files listed below from this checkout, then upload them together when prompted. The notebook does not rely on an older GitHub checkout. Internet is required for a Julia download if it is missing, and for installing any missing Python packages. `/content` is temporary: download the final ZIP.


In [ ]:
from pathlib import Path
import os, tempfile
from google.colab import files

required = {'cv_a100_validation.py', 'cv_a100_kernels.py', 'cv_photonics_benchmark.py', 'cv_photonics_julia.jl'}
print('Upload these four files from the repository experiments/ directory:', ', '.join(sorted(required)))
source_dir = Path(tempfile.mkdtemp(prefix='cv_photonics_source_', dir='/content'))
previous_directory = Path.cwd()
try:
    os.chdir(source_dir)
    uploaded = files.upload()
finally:
    os.chdir(previous_directory)
if set(uploaded) != required:
    raise ValueError(f'Upload exactly the four named files. Received: {sorted(uploaded)}')
for name, data in uploaded.items():
    (source_dir / name).write_bytes(data)
print('Uploaded sources:', sorted(p.name for p in source_dir.iterdir()))


In [ ]:
import importlib.util, subprocess, sys
packages = {'numpy': 'numpy', 'torch': 'torch', 'jax': 'jax[cuda12]', 'triton': 'triton'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:', missing, flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
    import importlib
    importlib.invalidate_caches()
import torch, jax, triton, numpy
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot access CUDA. Select a GPU runtime and restart.')
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__, 'JAX:', jax.__version__, 'Triton:', triton.__version__, 'NumPy:', numpy.__version__)
if 'A100' not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError('This validation requires an A100 runtime.')
if not any(d.platform == 'gpu' for d in jax.devices()):
    raise RuntimeError('JAX is installed but cannot access the GPU. Install a CUDA-compatible JAX, restart the runtime and rerun.')
print('JAX devices:', jax.devices())


In [ ]:
import hashlib, shutil, subprocess, urllib.request
julia = shutil.which('julia')
if not julia:
    version = '1.10.10'
    filename = f'julia-{version}-linux-x86_64.tar.gz'
    base = 'https://julialang-s3.julialang.org/bin'
    print('Downloading Julia', version, '(this can take a few minutes)...', flush=True)
    with urllib.request.urlopen(f'{base}/checksums/julia-{version}.sha256', timeout=120) as response:
        checksums = response.read().decode().splitlines()
    matches = [line.split()[0] for line in checksums if line.split()[-1].lstrip('*') == filename]
    if len(matches) != 1:
        raise RuntimeError('Official checksum for Julia archive not found uniquely.')
    archive = Path('/content') / filename
    with urllib.request.urlopen(f'{base}/linux/x64/1.10/{filename}', timeout=120) as response, archive.open('wb') as out:
        shutil.copyfileobj(response, out)
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != matches[0]:
        raise RuntimeError('Julia archive checksum mismatch; refusing to extract.')
    subprocess.run(['tar', '-xzf', str(archive), '-C', '/content'], check=True)
    julia = f'/content/julia-{version}/bin/julia'
print('Julia:', subprocess.run([julia, '--version'], check=True, capture_output=True, text=True).stdout.strip())


## Run three independent seeds

The first GPU compilation can take a few minutes. The command prints each report as it finishes and stops on a backend mismatch or error. `shots` controls Monte Carlo precision, not the size of a quantum computer. Configuration defaults are the same as the repository protocol. The output includes no speed or physical-energy claim.


In [ ]:
from datetime import datetime, timezone
import shutil
shots = 50000
seeds = [108, 109, 110]
run_dir = Path('/content/CV_Photonics_A100') / datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
run_dir.mkdir(parents=True, exist_ok=False)
for name in required:
    shutil.copy2(source_dir / name, run_dir / name)
for seed in seeds:
    print(f'Running seed {seed}, {shots} shots...', flush=True)
    subprocess.run([sys.executable, '-u', str(source_dir / 'cv_a100_validation.py'),
                    '--output-dir', str(run_dir / f'seed_{seed}'), '--shots', str(shots),
                    '--seed', str(seed), '--julia', julia], check=True)
print('All runs completed:', run_dir)


In [ ]:
import json
from google.colab import files
for path in sorted(run_dir.glob('seed_*/cv_a100_validation.json')):
    data = json.loads(path.read_text())
    print(path.parent.name, 'status:', data['status'])
    for name, result in data['results'].items():
        print(name, 'sample MSE:', round(result['sample_mse'], 6), 'theory:', round(result['analytic_mse'], 6))
    print('Backend agreement:', data['backend_checks'])
zip_path = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
print('Download:', zip_path)
files.download(zip_path)
